# RAG Application (Large model Mistral/Zephyr Edition)

This notebook implements a **Retrieval-Augmented Generation (RAG)** pipeline using
[HuggingFaceH4/zephyr-7b-beta](https://huggingface.co/HuggingFaceH4/zephyr-7b-beta)
— a high-quality Mistral-7B fine-tune that does **not** require a gated HuggingFace token.

The notebook automatically generates all required dependency files directly in the Colab
file system.

---
## ⚠️ Crucial First Steps

1. Open the **Files pane** (folder icon on the left).
2. **Upload your PDF file** (`HBR_How_Apple_Is_Organized_For_Innovation.pdf`).  
   *(Alternatively upload `sample_data.pdf` if you are testing with the provided dataset.)*
3. Ensure you are using a **T4 GPU**
   (`Runtime ▸ Change runtime type ▸ Hardware accelerator ▸ T4 GPU`).
4. Run all the cells below sequentially!

---
## Architecture Overview

```
PDF Document
     │
     ▼
[PyMuPDFLoader]  ──►  Raw pages
     │
     ▼
[RecursiveCharacterTextSplitter]  ──►  Chunks (1024 tokens, 20 overlap)
     │
     ▼
[HuggingFaceEmbeddings: mxbai-embed-large-v1]  ──►  Dense vectors (1024-dim)
     │
     ▼
[ChromaDB Vector Store]  ──►  Persisted similarity index (vector_db_1024/)
          │
    User Query ──► get_context() → top-k=3 chunks
                        │
                        ▼
                [Zephyr-7B-beta LLM]  ──►  Grounded Answer
                        │
                        ▼
              [Groundedness + Relevance Rater (LLM-as-a-Judge)]
```


## Write Dependency Files

All helper Python files are written into the Colab file system using `%%writefile` so the notebook is fully self-contained.

In [ ]:
%%writefile requirements.txt
huggingface_hub
tiktoken
pymupdf
langchain-community
langchain
langchain-chroma
langchain-text-splitters
langchain-huggingface
transformers
accelerate
bitsandbytes
pandas
sentence-transformers
python-dotenv


In [ ]:
%%writefile prompt_templates.py
"""
prompt_templates.py
-------------------
Centralised prompt strings used by the RAG pipeline.

Exported names
--------------
QNA_SYSTEM_MESSAGE              – system role prompt for the Q&A chain
QNA_USER_MESSAGE_TEMPLATE       – user turn with {context} and {question}
GROUNDEDNESS_RATER_SYSTEM_MESSAGE – rubric for groundedness evaluation
RELEVANCE_RATER_SYSTEM_MESSAGE    – rubric for relevance evaluation
EVAL_USER_MESSAGE_TEMPLATE        – shared user turn for both raters
"""

# ── Q&A prompts ────────────────────────────────────────────────────────────────
QNA_SYSTEM_MESSAGE = """You are a highly accurate and concise question-answering assistant.
Your sole purpose is to answer questions ONLY based on the provided CONTEXT.
You will be given a CONTEXT and a QUESTION.
You must follow these strict rules when answering:
RULES:
1. If the answer to the question IS PRESENT in the provided CONTEXT, answer concisely.
2. If the answer to the question IS NOT PRESENT in the provided CONTEXT, respond "I don't know". No additional information, no apologies, no elaborations.
3. DO NOT use any external knowledge. Rely strictly on the provided CONTEXT.
4. Do not rephrase the question in your answer.
Strictly adhere to these rules."""

QNA_USER_MESSAGE_TEMPLATE = """###Context
Here are some documents that are relevant to the question mentioned below.
{context}

###Question
{question}"""

# ── Groundedness rater ─────────────────────────────────────────────────────────
GROUNDEDNESS_RATER_SYSTEM_MESSAGE = """You are tasked with rating AI generated answers to questions posed by users.
You will be presented a question, context used by the AI system to generate the answer and an AI generated answer to the question.
In the input, the question will begin with ###Question, the context will begin with ###Context while the AI generated answer will begin with ###Answer.
Evaluation criteria:
The task is to judge the extent to which the metric is followed by the answer.
1 - The metric is not followed at all
2 - The metric is followed only to a limited extent
3 - The metric is followed to a good extent
4 - The metric is followed mostly
5 - The metric is followed completely
Metric:
The answer should be derived only from the information presented in the context
Instructions:
1. First write down the steps that are needed to evaluate the answer as per the metric.
2. Give a step-by-step explanation if the answer adheres to the metric considering the question and context as the input.
3. Next, evaluate the extent to which the metric is followed.
4. Use the previous information to rate the answer using the evaluation criteria and assign a score."""

# ── Relevance rater ────────────────────────────────────────────────────────────
RELEVANCE_RATER_SYSTEM_MESSAGE = """You are tasked with rating AI generated answers to questions posed by users.
You will be presented a question, context used by the AI system to generate the answer and an AI generated answer to the question.
In the input, the question will begin with ###Question, the context will begin with ###Context while the AI generated answer will begin with ###Answer.
Evaluation criteria:
The task is to judge the extent to which the metric is followed by the answer.
1 - The metric is not followed at all
2 - The metric is followed only to a limited extent
3 - The metric is followed to a good extent
4 - The metric is followed mostly
5 - The metric is followed completely
Metric:
Relevance measures how well the answer addresses the main aspects of the question, based on the context.
Consider whether all and only the important aspects are contained in the answer when evaluating relevance.
Instructions:
1. First write down the steps that are needed to evaluate the context as per the metric.
2. Give a step-by-step explanation if the context adheres to the metric considering the question as the input.
3. Next, evaluate the extent to which the metric is followed.
4. Use the previous information to rate the context using the evaluation criteria and assign a score."""

# ── Shared evaluation user-turn template ───────────────────────────────────────
EVAL_USER_MESSAGE_TEMPLATE = """###Question
{question}

###Context
{context}

###Answer
{answer}"""


In [ ]:
%%writefile config.py
"""
config.py
---------
All tuneable hyper-parameters and file paths live here.
Edit this file to change models, chunk sizes, or file paths.
"""

import os
from dotenv import load_dotenv

load_dotenv()

# ── Model selection ────────────────────────────────────────────────────────────
# Zephyr-7B-beta is high-quality and NOT gated – no HF token required.
# Uncomment an alternative if you have configured HF_TOKEN:
HUGGINGFACE_MODEL = "HuggingFaceH4/zephyr-7b-beta"
# HUGGINGFACE_MODEL = "mistralai/Mistral-7B-Instruct-v0.2"
# HUGGINGFACE_MODEL = "meta-llama/Meta-Llama-3-8B-Instruct"

DEFAULT_MODEL_NAME = HUGGINGFACE_MODEL

# ── Data & vector-store paths ──────────────────────────────────────────────────
APPLE_PDF_PATH = "HBR_How_Apple_Is_Organized_For_Innovation.pdf"
VECTOR_DB_DIR  = "vector_db_1024"

# ── Text-splitting settings ────────────────────────────────────────────────────
CHUNK_SIZE    = 1024
CHUNK_OVERLAP = 20
ENCODING_NAME = "cl100k_base"

# ── Embedding model ────────────────────────────────────────────────────────────
EMBEDDING_MODEL_NAME = "mixedbread-ai/mxbai-embed-large-v1"
DEFAULT_K_RETRIEVER  = 3

# ── LLM generation hyper-parameters ───────────────────────────────────────────
DEFAULT_MAX_TOKENS  = 8192
DEFAULT_TEMPERATURE = 0.1
DEFAULT_TOP_P       = 0.9
DEFAULT_TOP_K       = 40


In [ ]:
%%writefile functions.py
"""
functions.py
------------
RAG_LLM class – full Retrieval-Augmented Generation pipeline.

Method summary
--------------
__init__                   – initialise attributes, load the LLM
_initialize_hf_model       – download & load Zephyr in 4-bit NF4 quantisation
set_model                  – hot-swap the LLM at runtime
load_data                  – parse a PDF page-by-page with PyMuPDF
chunk_data                 – split pages into tiktoken-sized chunks
create_embeddings          – load the mxbai-embed-large-v1 embedding model
setup_vector_database      – create (or reload) ChromaDB + retriever in one call
get_context                – retrieve top-k chunks for a query string
create_rag_prompt          – build [system, user] message list for Q&A
generate_llm_response      – invoke the HF pipeline and return stripped text
get_answer                 – full RAG chain: retrieve → prompt → generate
create_groundedness_prompt – build eval prompt for groundedness metric
create_relevance_prompt    – build eval prompt for relevance metric
rate_groundedness          – LLM judge: score groundedness 1-5
rate_relevance             – LLM judge: score relevance 1-5
rate_answer                – return {"groundedness": ..., "relevance": ...}
calculate_rating           – convenience: answer + rate + formatted print
"""

import os
import torch

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    pipeline,
    BitsAndBytesConfig,
)
from langchain_huggingface import HuggingFacePipeline, HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_chroma import Chroma

from config import (
    APPLE_PDF_PATH,
    VECTOR_DB_DIR,
    CHUNK_SIZE,
    CHUNK_OVERLAP,
    ENCODING_NAME,
    EMBEDDING_MODEL_NAME,
    DEFAULT_K_RETRIEVER,
    DEFAULT_MAX_TOKENS,
    DEFAULT_TEMPERATURE,
    DEFAULT_TOP_P,
    DEFAULT_TOP_K,
    DEFAULT_MODEL_NAME,
)
from prompt_templates import (
    QNA_SYSTEM_MESSAGE,
    QNA_USER_MESSAGE_TEMPLATE,
    GROUNDEDNESS_RATER_SYSTEM_MESSAGE,
    RELEVANCE_RATER_SYSTEM_MESSAGE,
    EVAL_USER_MESSAGE_TEMPLATE,
)


class RAG_LLM:
    """End-to-end RAG pipeline backed by a local HuggingFace causal LLM."""

    # ── Construction ─────────────────────────────────────────────────────────────
    def __init__(self):
        self.model_name      = DEFAULT_MODEL_NAME
        print(f"Default model for this instance is set to: \'{self.model_name}\'")

        self.tokenizer       = None
        self.llm             = None
        self.documents       = None
        self.document_chunks = None
        self.embedding_model = None
        self.vectorstore     = None
        self.retriever       = None

        self._initialize_hf_model()
        print("RAG_LLM initialized.")

    # ── 1. LLM loading ───────────────────────────────────────────────────────────
    def _initialize_hf_model(self):
        """
        Download tokenizer and causal LM, quantise to 4-bit NF4, build a
        HuggingFace text-generation pipeline, wrap in LangChain object.
        """
        print(
            f"Initializing Hugging Face Pipeline for {self.model_name}"
            " (This may take a moment)..."
        )
        try:
            self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
            if self.tokenizer.pad_token is None:
                self.tokenizer.pad_token = self.tokenizer.eos_token

            bnb_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.bfloat16,
                bnb_4bit_use_double_quant=False,
            )

            model = AutoModelForCausalLM.from_pretrained(
                self.model_name,
                device_map="auto",
                quantization_config=bnb_config,
            )

            pipe = pipeline(
                "text-generation",
                model=model,
                tokenizer=self.tokenizer,
                max_new_tokens=DEFAULT_MAX_TOKENS,
                temperature=DEFAULT_TEMPERATURE,
                top_p=DEFAULT_TOP_P,
                top_k=DEFAULT_TOP_K,
                do_sample=True if DEFAULT_TEMPERATURE > 0 else False,
                return_full_text=False,
            )
            self.llm = HuggingFacePipeline(pipeline=pipe)
            print("Hugging Face model loaded successfully.")

        except Exception as e:
            print(f"Error loading Hugging Face model: {e}")
            self.llm = None

    # ── 2. Model hot-swap ────────────────────────────────────────────────────────
    def set_model(self, model_name: str):
        """Switch to a different HuggingFace model at runtime."""
        if self.model_name != model_name:
            self.model_name = model_name
            print(f"Default model for this instance has been changed to: \'{self.model_name}\'")
            self._initialize_hf_model()
        else:
            print(f"Model is already set to \'{self.model_name}\'")

    # ── 3. Data loading ──────────────────────────────────────────────────────────
    def load_data(self, pdf_path: str = APPLE_PDF_PATH):
        """Load a PDF page-by-page using PyMuPDF; returns list[Document]."""
        print(f"Loading data from: {pdf_path}")
        try:
            pdf_loader = PyMuPDFLoader(pdf_path)
            self.documents = pdf_loader.load()
            print(f"Successfully loaded {len(self.documents)} pages.")
            return self.documents
        except Exception as e:
            print(f"Error loading PDF data: {e}")
            return None

    # ── 4. Chunking ──────────────────────────────────────────────────────────────
    def chunk_data(
        self,
        documents: list,
        chunk_size: int    = CHUNK_SIZE,
        chunk_overlap: int = CHUNK_OVERLAP,
        encoding_name: str = ENCODING_NAME,
    ):
        """Split Document pages into overlapping token-level chunks."""
        if not documents:
            print("No documents provided for chunking.")
            return None

        print(f"Chunking data with chunk_size={chunk_size}, chunk_overlap={chunk_overlap}")
        try:
            text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
                encoding_name=encoding_name,
                chunk_size=chunk_size,
                chunk_overlap=chunk_overlap,
            )
            self.document_chunks = text_splitter.split_documents(documents)
            print(f"Created {len(self.document_chunks)} chunks.")
            return self.document_chunks
        except Exception as e:
            print(f"Error chunking data: {e}")
            return None

    # ── 5. Embeddings ────────────────────────────────────────────────────────────
    def create_embeddings(self, model_name: str = EMBEDDING_MODEL_NAME):
        """Load the HuggingFace sentence-embedding model (mxbai-embed-large-v1)."""
        print(f"Initializing embedding model: {model_name}")
        try:
            self.embedding_model = HuggingFaceEmbeddings(model_name=model_name)
            print("Embedding model initialized successfully.")
        except Exception as e:
            print(f"Error initializing embedding model: {e}")

    # ── 6. Vector database ───────────────────────────────────────────────────────
    def setup_vector_database(
        self,
        document_chunks: list = None,
        persist_directory: str = VECTOR_DB_DIR,
    ):
        """
        Create a new ChromaDB from document chunks, or reload an existing one.
        Attaches a similarity-search retriever after setup.
        """
        if not self.embedding_model:
            print("Embedding model not initialized. Please call create_embeddings() first.")
            return

        if not document_chunks and not self.document_chunks:
            print("No document chunks provided or available to set up vector database.")
            return

        chunks_to_use = document_chunks if document_chunks is not None else self.document_chunks
        print(f"Setting up vector database in: {persist_directory}")

        try:
            if not os.path.exists(persist_directory) or not os.listdir(persist_directory):
                os.makedirs(persist_directory, exist_ok=True)
                self.vectorstore = Chroma.from_documents(
                    chunks_to_use,
                    self.embedding_model,
                    persist_directory=persist_directory,
                )
                print("Vector database created and persisted.")
            else:
                self.vectorstore = Chroma(
                    persist_directory=persist_directory,
                    embedding_function=self.embedding_model,
                )
                print("Vector database loaded from existing directory.")

            self.retriever = self.vectorstore.as_retriever(
                search_type="similarity",
                search_kwargs={"k": DEFAULT_K_RETRIEVER},
            )
            print("Retriever initialized.")

        except Exception as e:
            print(f"Error setting up vector database: {e}")

    # ── 7. Context retrieval ─────────────────────────────────────────────────────
    def get_context(self, user_input: str, k: int = DEFAULT_K_RETRIEVER) -> str:
        """Retrieve top-k relevant chunks and join them into a single string."""
        if not self.retriever:
            print("Retriever not initialized. Please set up the vector database first.")
            return ""

        print(f"Retrieving {k} relevant documents for the query.")
        try:
            relevant_document_chunks = self.retriever.invoke(user_input)
            context_list = [d.page_content for d in relevant_document_chunks]
            return ". ".join(context_list)
        except Exception as e:
            print(f"Error getting context: {e}")
            return ""

    # ── 8. RAG prompt builder ────────────────────────────────────────────────────
    def create_rag_prompt(self, question: str, k: int = DEFAULT_K_RETRIEVER) -> list:
        """Build the [system, user] message list for the Q&A chain."""
        context_for_query = self.get_context(question, k=k)
        if not context_for_query:
            print("Could not retrieve context for the prompt.")
            return [{"role": "user", "content": question}]

        user_message = QNA_USER_MESSAGE_TEMPLATE.replace("{context}", context_for_query)
        user_message = user_message.replace("{question}", question)

        prompt = [
            {"role": "system", "content": QNA_SYSTEM_MESSAGE},
            {"role": "user",   "content": user_message},
        ]
        print("RAG prompt created.")
        return prompt

    # ── 9. LLM inference ─────────────────────────────────────────────────────────
    def generate_llm_response(
        self,
        prompt: list,
        model: str         = None,
        max_tokens: int    = DEFAULT_MAX_TOKENS,
        temperature: float = DEFAULT_TEMPERATURE,
        top_p: float       = DEFAULT_TOP_P,
        top_k: int         = DEFAULT_TOP_K,
        **kwargs,
    ) -> str:
        """
        Format the message list with apply_chat_template (or fallback),
        then invoke the HuggingFace pipeline.
        """
        model_to_use = model if model is not None else self.model_name
        if model_to_use != self.model_name:
            self.set_model(model_to_use)

        print(f"Generating LLM response using model: {self.model_name}")

        if self.llm is None:
            return "Error: Hugging Face model is not initialized. Check for loading errors above."

        try:
            if hasattr(self.tokenizer, "chat_template") and self.tokenizer.chat_template:
                formatted_prompt = self.tokenizer.apply_chat_template(
                    prompt, tokenize=False, add_generation_prompt=True
                )
            else:
                system_message   = prompt[0]["content"] if prompt and prompt[0]["role"] == "system" else ""
                user_message     = prompt[1]["content"] if len(prompt) > 1 and prompt[1]["role"] == "user" else ""
                formatted_prompt = f"System: {system_message}\nUser: {user_message}\nAssistant:"

            invoke_kwargs = {}
            if max_tokens != DEFAULT_MAX_TOKENS:
                invoke_kwargs["max_new_tokens"] = max_tokens
            if temperature != DEFAULT_TEMPERATURE:
                invoke_kwargs["temperature"] = temperature
                invoke_kwargs["do_sample"]   = True if temperature > 0 else False
            if top_p != DEFAULT_TOP_P:
                invoke_kwargs["top_p"] = top_p
            if top_k != DEFAULT_TOP_K:
                invoke_kwargs["top_k"] = top_k

            message = self.llm.invoke(formatted_prompt, **invoke_kwargs)
            print("Hugging Face response generated.")
            return message.strip()

        except Exception as e:
            return f"Sorry, I encountered an error with Hugging Face Inference: \n {e}"

    # ── 10. Full RAG Q&A ─────────────────────────────────────────────────────────
    def get_answer(self, user_input: str, k: int = DEFAULT_K_RETRIEVER, **llm_kwargs) -> str:
        """End-to-end RAG: retrieve context → build prompt → generate answer."""
        rag_prompt = self.create_rag_prompt(user_input, k=k)
        if not rag_prompt:
            print("Failed to create RAG prompt. Attempting to answer without context.")
            return self.generate_llm_response(
                [{"role": "user", "content": user_input}], **llm_kwargs
            )
        return self.generate_llm_response(rag_prompt, **llm_kwargs)

    # ── 11. Evaluation prompt builders ───────────────────────────────────────────
    def create_groundedness_prompt(
        self, question: str, answer: str, k: int = DEFAULT_K_RETRIEVER
    ):
        """Build the [system, user] prompt for groundedness scoring."""
        context_for_query = self.get_context(question, k=k)
        if not context_for_query:
            return None
        user_message = EVAL_USER_MESSAGE_TEMPLATE.replace("{context}", context_for_query)
        user_message = user_message.replace("{question}", question)
        user_message = user_message.replace("{answer}", answer)
        prompt = [
            {"role": "system", "content": GROUNDEDNESS_RATER_SYSTEM_MESSAGE},
            {"role": "user",   "content": user_message},
        ]
        return prompt

    def create_relevance_prompt(
        self, question: str, answer: str, k: int = DEFAULT_K_RETRIEVER
    ):
        """Build the [system, user] prompt for relevance scoring."""
        context_for_query = self.get_context(question, k=k)
        if not context_for_query:
            return None
        user_message = EVAL_USER_MESSAGE_TEMPLATE.replace("{context}", context_for_query)
        user_message = user_message.replace("{question}", question)
        user_message = user_message.replace("{answer}", answer)
        prompt = [
            {"role": "system", "content": RELEVANCE_RATER_SYSTEM_MESSAGE},
            {"role": "user",   "content": user_message},
        ]
        return prompt

    # ── 12. Evaluation scorers ────────────────────────────────────────────────────
    def rate_groundedness(
        self, question: str, answer: str, k: int = DEFAULT_K_RETRIEVER, **llm_kwargs
    ) -> str:
        """Score the answer on groundedness (1-5) using the LLM as judge."""
        print("Rating groundedness...")
        prompt = self.create_groundedness_prompt(question, answer, k=k)
        if not prompt:
            return "Groundedness evaluation failed: context not found."
        return self.generate_llm_response(prompt, max_tokens=200, temperature=0.1, **llm_kwargs)

    def rate_relevance(
        self, question: str, answer: str, k: int = DEFAULT_K_RETRIEVER, **llm_kwargs
    ) -> str:
        """Score the answer on relevance (1-5) using the LLM as judge."""
        print("Rating relevance...")
        prompt = self.create_relevance_prompt(question, answer, k=k)
        if not prompt:
            return "Relevance evaluation failed: context not found."
        return self.generate_llm_response(prompt, max_tokens=200, temperature=0.1, **llm_kwargs)

    def rate_answer(
        self, question: str, answer: str, k: int = DEFAULT_K_RETRIEVER, **llm_kwargs
    ) -> dict:
        """Return dict with groundedness and relevance ratings."""
        print("Rating overall answer quality (groundedness and relevance)...")
        groundedness_response = self.rate_groundedness(question, answer, k=k, **llm_kwargs)
        relevance_response    = self.rate_relevance(question, answer, k=k, **llm_kwargs)
        return {"groundedness": groundedness_response, "relevance": relevance_response}

    # ── 13. Convenience wrapper ───────────────────────────────────────────────────
    def calculate_rating(
        self, question: str, k: int = DEFAULT_K_RETRIEVER, **llm_kwargs
    ):
        """
        Full pipeline: generate an answer, rate it on groundedness and relevance,
        then print a formatted summary.
        """
        print(f"\n---- Calculating Ratings for Question: \'{question}\' ----")
        answer = self.get_answer(question, k=k, **llm_kwargs)
        rating = self.rate_answer(question, answer, k=k, **llm_kwargs)
        print("\n---- Results ----")
        print("\nQuestion: \n", question)
        print("\nAnswer: \n", answer)
        print("\nGroundedness Rating: \n", rating["groundedness"])
        print("\nRelevance Rating: \n", rating["relevance"])
        print("--------------------------------------------------")


## 1. Setup and Library Installation

Installing dependencies...

In [ ]:
!pip install -r requirements.txt
print("\n\nAll specified packages installed (or upgraded) successfully!")
print("Ready to initialize the Hugging Face model.")


## 2. (Optional) Hugging Face Login

If you edited `config.py` to use a gated model like `mistralai/Mistral-7B-Instruct-v0.2`
or `meta-llama/Meta-Llama-3-8B-Instruct`, you must log in below. Otherwise, **skip this cell**.


In [ ]:
from huggingface_hub import notebook_login
# notebook_login()  # Uncomment this to login via interactive prompt


## 3. Import Necessary Modules

We use `importlib.reload` so that any edits to `functions.py` are picked up
without restarting the Colab runtime.


In [ ]:
import importlib
import functions          # Import the functions module first
from functions import RAG_LLM
from config import APPLE_PDF_PATH, DEFAULT_MODEL_NAME, HUGGINGFACE_MODEL
import os

# Force-reload the functions module to pick up any recent changes
importlib.reload(functions)
from functions import RAG_LLM  # Re-import RAG_LLM from the reloaded module

print("Modules imported successfully.")


## 4. Initialize the RAG_LLM System

This step will download and load the Hugging Face model into memory.

> ⏱️ **Expected time:** 3–8 minutes on first run (model weights are cached afterwards).  
> Zephyr-7B-beta is ~14.5 GB in full precision but only ~4–5 GB VRAM in NF4 4-bit form.


In [ ]:
rag_system = RAG_LLM()
print("RAG_LLM system initialized.")


## 5. Load the PDF Document

`load_data()` uses **PyMuPDF** (via LangChain's `PyMuPDFLoader`) to parse the PDF
page-by-page. Each page becomes a separate LangChain `Document` with text and metadata.

> Make sure you have uploaded `HBR_How_Apple_Is_Organized_For_Innovation.pdf`  
> (or `sample_data.pdf`) to the Colab file system before running this cell.


In [ ]:
documents = rag_system.load_data(pdf_path=APPLE_PDF_PATH)
if not documents:
    print("Failed to load documents. Please check the PDF path and file existence.")
else:
    print("PDF document loaded successfully.")


## 6. Chunk the Loaded Data

Large documents are split into smaller **chunks** before embedding because:
- Embedding models have a fixed maximum input length.
- Shorter, focused chunks improve retrieval precision.

`RecursiveCharacterTextSplitter.from_tiktoken_encoder` counts tokens with the same
BPE tokeniser as GPT-4, ensuring chunks respect the 1024-token limit.
A 20-token overlap preserves context at chunk boundaries.

> The 11-page Apple article produces **~16 chunks**.


In [ ]:
document_chunks = rag_system.chunk_data(documents)
if not document_chunks:
    print("Failed to chunk documents.")
else:
    print("Documents chunked successfully.")


## 7. Create Embedding Model

We load **mixedbread-ai/mxbai-embed-large-v1**, a 335M-parameter bi-encoder that:
- Produces 1024-dimensional dense vectors.
- Ranks consistently in the top-5 on the [MTEB leaderboard](https://huggingface.co/spaces/mteb/leaderboard).
- Supports Matryoshka Representation Learning (MRL).

> ⏱️ Downloads ~670 MB on first run; cached afterwards.


In [ ]:
rag_system.create_embeddings()
if not rag_system.embedding_model:
    print("Failed to create embedding model.")
else:
    print("Embedding model created successfully.")


## 8. Set Up the Vector Database

`setup_vector_database()` handles both first-run and subsequent runs automatically:

| Situation | Action |
|---|---|
| `vector_db_1024/` **does not exist** or is empty | Embed all chunks → save to disk |
| `vector_db_1024/` **already exists** with data | Load existing vectors from disk (fast) |

After setup, `rag_system.retriever` is ready for similarity search.


In [ ]:
rag_system.setup_vector_database(document_chunks=document_chunks)
if not rag_system.vectorstore:
    print("Failed to set up vector database.")
else:
    print("Vector database set up and retriever initialized.")


## 9. Demonstrate Question Answering with RAG

`get_answer()` runs the full RAG chain:
1. **Retrieve** the top-3 most relevant chunks via `get_context()`.
2. **Format** the system + user prompt with the retrieved context.
3. **Generate** a grounded answer using Zephyr-7B-beta.


In [ ]:
# Confirm the current model (safe to call multiple times)
rag_system.set_model(DEFAULT_MODEL_NAME)


In [ ]:
# Example Query 1
user_input_1 = "Who are the authors of this article and who published this article ?"
print(f"\nQuery 1: {user_input_1}")
llm_response_1 = rag_system.get_answer(user_input_1)
print(f"Response 1: \n{llm_response_1}")


In [ ]:
# Example Query 2
user_input_2 = "List down the three leadership characteristics in bulleted points and explain each one of the characteristics under two lines."
print(f"\nQuery 2: {user_input_2}")
llm_response_2 = rag_system.get_answer(user_input_2, max_tokens=150, temperature=0.1)
print(f"Response 2: \n{llm_response_2}")


In [ ]:
# Example Query 3
user_input_3 = "Can you explain specific examples from the article where Apple's approach to leadership has led to successful innovations?"
print(f"\nQuery 3: {user_input_3}")
llm_response_3 = rag_system.get_answer(user_input_3)
print(f"Response 3: \n{llm_response_3}")


## 10. Demonstrate Output Evaluation (LLM-as-a-Judge)

We evaluate each answer on two axes using the same Zephyr model as judge:

| Metric | Definition | Scale |
|---|---|---|
| **Groundedness** | Is the answer derived *only* from the retrieved context? | 1–5 |
| **Relevance**    | Does the answer address *all key aspects* of the question? | 1–5 |

`calculate_rating()` calls `get_answer()`, `rate_groundedness()`, and `rate_relevance()`
in sequence and prints a fully formatted summary (question → answer → both ratings).


In [ ]:
# Evaluate Query 1
user_input_1 = "Who are the authors of this article and who published this article ?"
print("\nEvaluating Query 1:")
rag_system.calculate_rating(question=user_input_1)


In [ ]:
# Evaluate Query 2
user_input_2 = "List down the three leadership characteristics in bulleted points and explain each one of the characteristics under two lines."
print("\nEvaluating Query 2:")
rag_system.calculate_rating(question=user_input_2)


In [ ]:
# Evaluate Query 3
user_input_3 = "Can you explain specific examples from the article where Apple's approach to leadership has led to successful innovations?"
print("\nEvaluating Query 3:")
rag_system.calculate_rating(question=user_input_3)


In [ ]:
# 20 out of 20 :D


## 11. Interactive Q&A Loop

Run this cell to ask your own questions about the document.
Type **`quit`** or **`exit`** to stop.


In [ ]:
print("RAG Q&A ready. Type 'quit' to exit.\n")

while True:
    user_input = input("Your question: ").strip()
    if not user_input:
        continue
    if user_input.lower() in ("quit", "exit", "q"):
        print("Session ended.")
        break
    response = rag_system.get_answer(user_input)
    print(f"\nAnswer: {response}\n")


## (Optional) Switch to a Different Model

You can hot-swap the LLM without rebuilding the vector store. Set your token first:

```python
import os
os.environ["HF_TOKEN"] = "hf_your_token_here"
```


In [ ]:
# Uncomment one line below to switch models (requires HF_TOKEN for gated models):
# rag_system.set_model("mistralai/Mistral-7B-Instruct-v0.2")
# rag_system.set_model("meta-llama/Meta-Llama-3-8B-Instruct")

print("Current model:", rag_system.model_name)
